# 06 — Soft Actor-Critic (SAC)

Soft Actor-Critic (Haarnoja et al., 2018) is the dominant algorithm for continuous control tasks. It combines off-policy learning (sample efficiency of DQN) with a stochastic policy (expressiveness of PPO) plus a maximum entropy objective that encourages exploration without separate ε-greedy noise.

## Maximum entropy RL

Standard RL maximises expected return: E[Σ γ^t R_t].

Maximum entropy RL adds an entropy bonus:
$$J(\pi) = \mathbb{E}\left[\sum_t \gamma^t \left(R_t + \alpha \mathcal{H}(\pi(\cdot|s_t))\right)\right]$$

Where H(π(·|s)) = -E[log π(a|s)] is the entropy of the policy at state s, and α (the temperature) controls the entropy-reward tradeoff.

**Why this matters**:
- **Exploration**: the entropy bonus pushes the policy to try diverse actions, preventing premature convergence to suboptimal deterministic policies
- **Robustness**: a stochastic policy is harder to exploit — useful in multi-agent settings and sim-to-real transfer
- **Multiple optima**: if several action sequences achieve similar rewards, the max-entropy policy captures all of them rather than arbitrarily committing to one

SAC learns the temperature α automatically via a Lagrangian dual on a target entropy constraint H_target = -dim(A).

## SAC architecture

SAC maintains five networks:

| Network | Role |
|---------|------|
| Actor π(a\|s;θ) | Outputs mean and log-std of a Gaussian; action sampled via reparameterisation trick |
| Critic Q₁(s,a;φ₁) | First Q-network (double Q trick to reduce overestimation) |
| Critic Q₂(s,a;φ₂) | Second Q-network |
| Target Q₁ (φ₁⁻) | Polyak-averaged copy of Q₁ for stable targets |
| Target Q₂ (φ₂⁻) | Polyak-averaged copy of Q₂ |

Polyak averaging: φ⁻ ← τφ + (1-τ)φ⁻ with τ≈0.005 — slower and smoother than periodic hard updates.

The reparameterisation trick (action = tanh(mean + std·ε), ε~N(0,I)) makes the actor loss differentiable w.r.t. the sampled action.

In [ ]:
# SAC implementation for continuous control
import torch, torch.nn as nn, torch.optim as optim
import torch.nn.functional as F
import numpy as np, random
from collections import deque
import gymnasium as gym
import matplotlib.pyplot as plt

torch.manual_seed(42); np.random.seed(42); random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LOG_STD_MIN, LOG_STD_MAX = -20, 2

class GaussianActor(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(obs_dim,hidden),nn.ReLU(),nn.Linear(hidden,hidden),nn.ReLU())
        self.mean = nn.Linear(hidden, act_dim)
        self.log_std = nn.Linear(hidden, act_dim)

    def forward(self, obs):
        feat = self.net(obs)
        mean = self.mean(feat)
        log_std = self.log_std(feat).clamp(LOG_STD_MIN, LOG_STD_MAX)
        return mean, log_std

    def sample(self, obs):
        mean, log_std = self(obs)
        std = log_std.exp()
        eps = torch.randn_like(mean)
        x = mean + std * eps
        action = torch.tanh(x)
        log_prob = (torch.distributions.Normal(mean, std).log_prob(x)
                    - torch.log(1 - action.pow(2) + 1e-6)).sum(dim=-1)
        return action, log_prob

class QNetwork(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim+act_dim,hidden),nn.ReLU(),
            nn.Linear(hidden,hidden),nn.ReLU(),
            nn.Linear(hidden,1)
        )
    def forward(self, obs, act): return self.net(torch.cat([obs,act],dim=-1)).squeeze(-1)

class SACAgent:
    def __init__(self, obs_dim, act_dim, lr=3e-4, gamma=0.99, tau=0.005,
                 buffer_size=100_000, batch_size=256, target_entropy=None):
        self.gamma = gamma; self.tau = tau; self.batch_size = batch_size
        self.actor = GaussianActor(obs_dim, act_dim).to(device)
        self.q1 = QNetwork(obs_dim, act_dim).to(device)
        self.q2 = QNetwork(obs_dim, act_dim).to(device)
        self.q1_target = QNetwork(obs_dim, act_dim).to(device)
        self.q2_target = QNetwork(obs_dim, act_dim).to(device)
        self.q1_target.load_state_dict(self.q1.state_dict())
        self.q2_target.load_state_dict(self.q2.state_dict())
        self.actor_opt = optim.Adam(self.actor.parameters(), lr=lr)
        self.q1_opt = optim.Adam(self.q1.parameters(), lr=lr)
        self.q2_opt = optim.Adam(self.q2.parameters(), lr=lr)
        self.target_entropy = target_entropy if target_entropy else -act_dim
        self.log_alpha = torch.zeros(1, requires_grad=True, device=device)
        self.alpha_opt = optim.Adam([self.log_alpha], lr=lr)
        self.buffer = deque(maxlen=buffer_size)

    @property
    def alpha(self): return self.log_alpha.exp().item()

    def select_action(self, obs, deterministic=False):
        obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
        with torch.no_grad():
            if deterministic:
                mean, _ = self.actor(obs_t)
                return torch.tanh(mean).squeeze(0).cpu().numpy()
            action, _ = self.actor.sample(obs_t)
            return action.squeeze(0).cpu().numpy()

    def update(self):
        if len(self.buffer) < self.batch_size: return {}
        batch = random.sample(self.buffer, self.batch_size)
        s,a,r,ns,dn = [torch.FloatTensor(np.array(x)).to(device) for x in zip(*batch)]

        with torch.no_grad():
            na, na_lp = self.actor.sample(ns)
            q1_next = self.q1_target(ns, na)
            q2_next = self.q2_target(ns, na)
            min_q_next = torch.min(q1_next, q2_next) - self.alpha * na_lp
            target_q = r + self.gamma * (1 - dn) * min_q_next

        for q, opt in [(self.q1, self.q1_opt), (self.q2, self.q2_opt)]:
            loss = F.mse_loss(q(s, a), target_q)
            opt.zero_grad(); loss.backward(); opt.step()

        new_a, new_lp = self.actor.sample(s)
        min_q = torch.min(self.q1(s, new_a), self.q2(s, new_a))
        actor_loss = (self.alpha * new_lp - min_q).mean()
        self.actor_opt.zero_grad(); actor_loss.backward(); self.actor_opt.step()

        alpha_loss = -(self.log_alpha * (new_lp + self.target_entropy).detach()).mean()
        self.alpha_opt.zero_grad(); alpha_loss.backward(); self.alpha_opt.step()

        for q, qt in [(self.q1, self.q1_target), (self.q2, self.q2_target)]:
            for p, pt in zip(q.parameters(), qt.parameters()):
                pt.data.copy_(self.tau * p.data + (1 - self.tau) * pt.data)

        return {"actor_loss": actor_loss.item(), "alpha": self.alpha}

env = gym.make("LunarLanderContinuous-v2")
obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.shape[0]
agent = SACAgent(obs_dim, act_dim)
episode_returns = []
alpha_history = []

print("Training SAC on LunarLanderContinuous-v2...")
for ep in range(300):
    obs, _ = env.reset(seed=ep)
    ep_return = 0
    for _ in range(1000):
        action = agent.select_action(obs)
        next_obs, reward, term, trunc, _ = env.step(action)
        done = term or trunc
        agent.buffer.append((obs, action, reward, next_obs, float(done)))
        info = agent.update()
        obs = next_obs; ep_return += reward
        if done: break
    episode_returns.append(ep_return)
    if info: alpha_history.append(info.get("alpha", 0))
    if (ep + 1) % 50 == 0:
        avg = np.mean(episode_returns[-50:])
        print(f"  Episode {ep+1:4d} | Avg return: {avg:7.1f} | alpha={agent.alpha:.4f}")

env.close()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
w = 20
ax1.plot(episode_returns, alpha=0.3, color='steelblue')
ax1.plot(np.convolve(episode_returns, np.ones(w)/w, 'valid'), color='steelblue', label=f'{w}-ep avg')
ax1.axhline(200, color='green', linestyle='--', alpha=0.7, label='Good performance')
ax1.set_xlabel('Episode'); ax1.set_ylabel('Return')
ax1.set_title('SAC on LunarLanderContinuous-v2', fontweight='bold'); ax1.legend()
ax2.plot(alpha_history, color='darkorange', alpha=0.7)
ax2.set_xlabel('Update step'); ax2.set_ylabel('Temperature alpha')
ax2.set_title('Automatic Entropy Temperature', fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/sac_lunar.png', dpi=100, bbox_inches='tight')
plt.show()


## SAC vs PPO — when to use which

| Property | PPO | SAC |
|---------|-----|-----|
| Action space | Discrete or continuous | Continuous |
| Sample efficiency | Lower (on-policy) | Higher (off-policy) |
| Hyperparameter sensitivity | Low | Moderate |
| Wall-clock time | Faster with parallel envs | Faster with single env + replay |
| Best for | Fast iteration, RLHF, discrete | Robotics, sim-to-real, sample efficiency matters |

In practice: start with PPO. Switch to SAC if sample efficiency is the bottleneck and actions are continuous.